Week 11 (09/05/2025) - Multilayer perceptrons

In [7]:
# Pytorch implementation of a multilayer perceptron.
import torch # Pytorch works using tensors, which can be implemented as n-dimensional arrays similar to numpy arrays
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn # for creating the neural netowork layers
import torchmetrics

# Start by importing the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the training data of MNIST and into a 'data' folder
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the testing data of MNIST and into a 'data' folder

# Define the structure of the MLP model through classes extending the Module class of nn.
device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility
class MyMLP(nn.Module):
    def __init__(self): # define the layers needed by the model
        super().__init__() # initialize the attributes of the parent class
        self.mlp = nn.Sequential(
            # Define the neural network's structure.
            nn.Linear(28*28,5), # connect the input layer (one input node per feature) to the hidden layer (five nodes)
            nn.Sigmoid(), # sigmoid activation function
            nn.Linear(5,5), # connect the first hidden layer to the second hidden layer
            nn.Sigmoid(), # sigmoid activation function
            nn.Linear(5,5), # connect the second hidden layer to the third hidden layer
            nn.Sigmoid(), # sigmoid activation function
            nn.Linear(5,10) # connect the third hidden layer to the output layer (ten nodes, one for each class)
        )
        self.flatten = nn.Flatten() # flatten the structure into a one-dimensional array
    def forward(self, x): # specify how the data goes inside the model by connecting the layers
        x = self.flatten(x) # convert the input image into a one-dimensional array
        logits = self.mlp(x) # perform a raw classification on the prepared data that will be normalized using the loss function
        return logits # note that the output is a probbaility tensor, so something of type logits = [0.1, 0.0, ..., 0.6]

# Create an instance of the MLP model.
model = MyMLP()

# Start by defining the hyperparameters of the model.
epochs = 2 # check the whole dataset twice
batch_size = 64 # check 64 data samples at once
learning_rate = 0.0001 # use a smaller learning rate in order to obtain more accurate results during the backpropagation phase

# Define the loss function for the classification model.
loss_function = nn.CrossEntropyLoss() # choose cross entropy loss

# Use a dataloader for processing the data in batches.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # dataloader for the training dataset
test_dataloader = DataLoader(test_data, batch_size=batch_size) # dataloader for the testing dataset

# Create the accuracy metric for estimating the goodness of the model.
metric = torchmetrics.Accuracy(task='multiclass', num_classes=10)

# Define the optimizer for gradient descent.
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # perform stochastic gradient descent on the parameters

# Define the training loop for the training stage.
def training_loop(dataloader, model, loss_function, optimizer):
    dataset_size = len(dataloader)
    # Start by loading the data batch from the disk.
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X) # get the predictions of the model
        # Compute the prediction error using the loss function.
        loss = loss_function(pred, y)
        # Perform backpropagation in order to minimize future losses.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss:{loss}, [{current} /{dataset_size}]")
            acc = metric(pred, y)
            print(f"Accuracy of the current batch: {acc}")
    # Print the final training accuracy.
    acc = metric.compute()
    print(f"Final training accuracy: {acc}")
    metric.reset() # reset for future loops

# Define the testing loop for the testing stage.
def testing_loop(dataloader, loss_function, model):
    # Remember to disable the weights update.
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            acc = metric(pred, y)
    acc = metric.compute()
    print(f"Final testing accuracy: {acc}")
    metric.reset()

# Perform the actual model training and testing.
for e in range(epochs):
    print(f"Epoch: {e}")
    training_loop(train_dataloader, model, loss_function, optimizer)
    testing_loop(test_dataloader, loss_function, model)
print("Done")

Epoch: 0
loss:2.3390166759490967, [64 /938]
Accuracy of the current batch: 0.125
loss:2.3157525062561035, [32064 /938]
Accuracy of the current batch: 0.09375
Final training accuracy: 0.109375
Final testing accuracy: 0.10100000351667404
Epoch: 1
loss:2.3378782272338867, [64 /938]
Accuracy of the current batch: 0.125
loss:2.3151893615722656, [32064 /938]
Accuracy of the current batch: 0.09375
Final training accuracy: 0.109375
Final testing accuracy: 0.10100000351667404
Done


An alternative approach to improve performance is given by the ReLU activation function.

In [10]:
# Pytorch implementation of a multilayer perceptron.
import torch # Pytorch works using tensors, which can be implemented as n-dimensional arrays similar to numpy arrays
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn # for creating the neural netowork layers
import torchmetrics

# Start by importing the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the training data of MNIST and into a 'data' folder
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the testing data of MNIST and into a 'data' folder

# Define the structure of the MLP model through classes extending the Module class of nn.
device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility
class MyMLP(nn.Module):
    def __init__(self): # define the layers needed by the model
        super().__init__() # initialize the attributes of the parent class
        self.mlp = nn.Sequential(
            # Define the neural network's structure.
            nn.Linear(28*28,5), # connect the input layer (one input node per feature) to the hidden layer (five nodes)
            nn.ReLU(), # sigmoid activation function
            nn.Linear(5,5), # connect the first hidden layer to the second hidden layer
            nn.ReLU(), # sigmoid activation function
            nn.Linear(5,5), # connect the second hidden layer to the third hidden layer
            nn.ReLU(), # sigmoid activation function
            nn.Linear(5,10) # connect the third hidden layer to the output layer (ten nodes, one for each class)
        )
        self.flatten = nn.Flatten() # flatten the structure into a one-dimensional array
    def forward(self, x): # specify how the data goes inside the model by connecting the layers
        x = self.flatten(x) # convert the input image into a one-dimensional array
        logits = self.mlp(x) # perform a raw classification on the prepared data that will be normalized using the loss function
        return logits # note that the output is a probbaility tensor, so something of type logits = [0.1, 0.0, ..., 0.6]

# Create an instance of the MLP model.
model = MyMLP()

# Start by defining the hyperparameters of the model.
epochs = 5 # check the whole dataset five times
batch_size = 64 # check 64 data samples at once
learning_rate = 0.0001 # use a smaller learning rate in order to obtain more accurate results during the backpropagation phase

# Define the loss function for the classification model.
loss_function = nn.CrossEntropyLoss() # choose cross entropy loss

# Use a dataloader for processing the data in batches.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # dataloader for the training dataset
test_dataloader = DataLoader(test_data, batch_size=batch_size) # dataloader for the testing dataset

# Create the accuracy metric for estimating the goodness of the model.
metric = torchmetrics.Accuracy(task='multiclass', num_classes=10)

# Define the optimizer for gradient descent.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) # use the AdamW model on the parameters

# Define the training loop for the training stage.
def training_loop(dataloader, model, loss_function, optimizer):
    dataset_size = len(dataloader)
    # Start by loading the data batch from the disk.
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X) # get the predictions of the model
        # Compute the prediction error using the loss function.
        loss = loss_function(pred, y)
        # Perform backpropagation in order to minimize future losses.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss:{loss}, [{current} /{dataset_size}]")
            acc = metric(pred, y)
            print(f"Accuracy of the current batch: {acc}")
    # Print the final training accuracy.
    acc = metric.compute()
    print(f"Final training accuracy: {acc}")
    metric.reset() # reset for future loops

# Define the testing loop for the testing stage.
def testing_loop(dataloader, loss_function, model):
    # Remember to disable the weights update.
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            acc = metric(pred, y)
    acc = metric.compute()
    print(f"Final testing accuracy: {acc}")
    metric.reset()

# Perform the actual model training and testing.
for e in range(epochs):
    print(f"Epoch: {e}")
    training_loop(train_dataloader, model, loss_function, optimizer)
    testing_loop(test_dataloader, loss_function, model)
print("Done")

Epoch: 0
loss:2.309253215789795, [64 /938]
Accuracy of the current batch: 0.140625
loss:2.3472158908843994, [32064 /938]
Accuracy of the current batch: 0.09375
Final training accuracy: 0.1171875
Final testing accuracy: 0.10090000182390213
Epoch: 1
loss:2.302572011947632, [64 /938]
Accuracy of the current batch: 0.140625
loss:2.335172414779663, [32064 /938]
Accuracy of the current batch: 0.09375
Final training accuracy: 0.1171875
Final testing accuracy: 0.10090000182390213
Epoch: 2
loss:2.29850435256958, [64 /938]
Accuracy of the current batch: 0.140625
loss:2.3265388011932373, [32064 /938]
Accuracy of the current batch: 0.09375
Final training accuracy: 0.1171875
Final testing accuracy: 0.10090000182390213
Epoch: 3
loss:2.2962377071380615, [64 /938]
Accuracy of the current batch: 0.140625
loss:2.3204400539398193, [32064 /938]
Accuracy of the current batch: 0.09375
Final training accuracy: 0.1171875
Final testing accuracy: 0.10090000182390213
Epoch: 4
loss:2.2951722145080566, [64 /938]
A

In [ ]:
# Pytorch implementation of a multilayer perceptron.
import torch # Pytorch works using tensors, which can be implemented as n-dimensional arrays similar to numpy arrays
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn # for creating the neural netowork layers
import torchmetrics

# Start by importing the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the training data of MNIST and into a 'data' folder
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the testing data of MNIST and into a 'data' folder

# Define the structure of the MLP model through classes extending the Module class of nn.
# Try now to remove one hidden layer.
device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility
class MyMLP(nn.Module):
    def __init__(self): # define the layers needed by the model
        super().__init__() # initialize the attributes of the parent class
        self.mlp = nn.Sequential(
            # Define the neural network's structure.
            nn.Linear(28*28,5), # connect the input layer (one input node per feature) to the hidden layer (five nodes)
            nn.ReLU(), # sigmoid activation function
            nn.Linear(5,5), # connect the first hidden layer to the second hidden layer
            nn.ReLU(), # sigmoid activation function
            # nn.Linear(5,5), # connect the second hidden layer to the third hidden layer
            # nn.ReLU(), # sigmoid activation function
            nn.Linear(5,10) # connect the third hidden layer to the output layer (ten nodes, one for each class)
        )
        self.flatten = nn.Flatten() # flatten the structure into a one-dimensional array
    def forward(self, x): # specify how the data goes inside the model by connecting the layers
        x = self.flatten(x) # convert the input image into a one-dimensional array
        logits = self.mlp(x) # perform a raw classification on the prepared data that will be normalized using the loss function
        return logits # note that the output is a probbaility tensor, so something of type logits = [0.1, 0.0, ..., 0.6]

# Create an instance of the MLP model.
model = MyMLP()

# Start by defining the hyperparameters of the model.
epochs = 5 # check the whole dataset five times
batch_size = 64 # check 64 data samples at once
learning_rate = 0.0001 # use a smaller learning rate in order to obtain more accurate results during the backpropagation phase

# Define the loss function for the classification model.
loss_function = nn.CrossEntropyLoss() # choose cross entropy loss

# Use a dataloader for processing the data in batches.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # dataloader for the training dataset
test_dataloader = DataLoader(test_data, batch_size=batch_size) # dataloader for the testing dataset

# Create the accuracy metric for estimating the goodness of the model.
metric = torchmetrics.Accuracy(task='multiclass', num_classes=10)

# Define the optimizer for gradient descent.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) # use the AdamW model on the parameters

# Define the training loop for the training stage.
def training_loop(dataloader, model, loss_function, optimizer):
    dataset_size = len(dataloader)
    # Start by loading the data batch from the disk.
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X) # get the predictions of the model
        # Compute the prediction error using the loss function.
        loss = loss_function(pred, y)
        # Perform backpropagation in order to minimize future losses.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss:{loss}, [{current} /{dataset_size}]")
            acc = metric(pred, y)
            print(f"Accuracy of the current batch: {acc}")
    # Print the final training accuracy.
    acc = metric.compute()
    print(f"Final training accuracy: {acc}")
    metric.reset() # reset for future loops

# Define the testing loop for the testing stage.
def testing_loop(dataloader, loss_function, model):
    # Remember to disable the weights update.
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            acc = metric(pred, y)
    acc = metric.compute()
    print(f"Final testing accuracy: {acc}")
    metric.reset()

# Perform the actual model training and testing.
for e in range(epochs):
    print(f"Epoch: {e}")
    training_loop(train_dataloader, model, loss_function, optimizer)
    testing_loop(test_dataloader, loss_function, model)
print("Done")

Epoch: 0
loss:2.309253215789795, [64 /938]
Accuracy of the current batch: 0.140625
loss:2.3472158908843994, [32064 /938]
Accuracy of the current batch: 0.09375
Final training accuracy: 0.1171875
Final testing accuracy: 0.10090000182390213
Epoch: 1
loss:2.302572011947632, [64 /938]
Accuracy of the current batch: 0.140625
loss:2.335172414779663, [32064 /938]
Accuracy of the current batch: 0.09375
Final training accuracy: 0.1171875
Final testing accuracy: 0.10090000182390213
Epoch: 2
loss:2.29850435256958, [64 /938]
Accuracy of the current batch: 0.140625
loss:2.3265388011932373, [32064 /938]
Accuracy of the current batch: 0.09375


In [12]:
# Pytorch implementation of a multilayer perceptron.
import torch # Pytorch works using tensors, which can be implemented as n-dimensional arrays similar to numpy arrays
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn # for creating the neural netowork layers
import torchmetrics

# Start by importing the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the training data of MNIST and into a 'data' folder
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the testing data of MNIST and into a 'data' folder

# Define the structure of the MLP model through classes extending the Module class of nn.
# Try now to remove one hidden layer.
device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility
class MyMLP(nn.Module):
    def __init__(self): # define the layers needed by the model
        super().__init__() # initialize the attributes of the parent class
        self.mlp = nn.Sequential(
            # Define the neural network's structure.
            nn.Linear(28*28,15), # connect the input layer (one input node per feature) to the hidden layer (five nodes)
            nn.ReLU(), # sigmoid activation function
            nn.Linear(15,15), # connect the first hidden layer to the second hidden layer
            nn.ReLU(), # sigmoid activation function
            # nn.Linear(5,5), # connect the second hidden layer to the third hidden layer
            # nn.ReLU(), # sigmoid activation function
            nn.Linear(15,10) # connect the third hidden layer to the output layer (ten nodes, one for each class)
        )
        self.flatten = nn.Flatten() # flatten the structure into a one-dimensional array
    def forward(self, x): # specify how the data goes inside the model by connecting the layers
        x = self.flatten(x) # convert the input image into a one-dimensional array
        logits = self.mlp(x) # perform a raw classification on the prepared data that will be normalized using the loss function
        return logits # note that the output is a probbaility tensor, so something of type logits = [0.1, 0.0, ..., 0.6]

# Create an instance of the MLP model.
model = MyMLP()

# Start by defining the hyperparameters of the model.
epochs = 5 # check the whole dataset five times
batch_size = 64 # check 64 data samples at once
learning_rate = 0.0001 # use a smaller learning rate in order to obtain more accurate results during the backpropagation phase

# Define the loss function for the classification model.
loss_function = nn.CrossEntropyLoss() # choose cross entropy loss

# Use a dataloader for processing the data in batches.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # dataloader for the training dataset
test_dataloader = DataLoader(test_data, batch_size=batch_size) # dataloader for the testing dataset

# Create the accuracy metric for estimating the goodness of the model.
metric = torchmetrics.Accuracy(task='multiclass', num_classes=10)

# Define the optimizer for gradient descent.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) # use the AdamW model on the parameters

# Define the training loop for the training stage.
def training_loop(dataloader, model, loss_function, optimizer):
    dataset_size = len(dataloader)
    # Start by loading the data batch from the disk.
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X) # get the predictions of the model
        # Compute the prediction error using the loss function.
        loss = loss_function(pred, y)
        # Perform backpropagation in order to minimize future losses.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss:{loss}, [{current} /{dataset_size}]")
            acc = metric(pred, y)
            print(f"Accuracy of the current batch: {acc}")
    # Print the final training accuracy.
    acc = metric.compute()
    print(f"Final training accuracy: {acc}")
    metric.reset() # reset for future loops

# Define the testing loop for the testing stage.
def testing_loop(dataloader, loss_function, model):
    # Remember to disable the weights update.
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            acc = metric(pred, y)
    acc = metric.compute()
    print(f"Final testing accuracy: {acc}")
    metric.reset()

# Perform the actual model training and testing.
for e in range(epochs):
    print(f"Epoch: {e}")
    training_loop(train_dataloader, model, loss_function, optimizer)
    testing_loop(test_dataloader, loss_function, model)
print("Done")

Epoch: 0
loss:2.3517446517944336, [64 /938]
Accuracy of the current batch: 0.0625
loss:1.7182341814041138, [32064 /938]
Accuracy of the current batch: 0.609375
Final training accuracy: 0.3359375
Final testing accuracy: 0.7476000189781189
Epoch: 1
loss:1.0818008184432983, [64 /938]
Accuracy of the current batch: 0.703125
loss:0.7539736032485962, [32064 /938]
Accuracy of the current batch: 0.796875
Final training accuracy: 0.75
Final testing accuracy: 0.8501999974250793
Epoch: 2
loss:0.6381168365478516, [64 /938]
Accuracy of the current batch: 0.796875
loss:0.52532958984375, [32064 /938]
Accuracy of the current batch: 0.828125
Final training accuracy: 0.8125
Final testing accuracy: 0.8756999969482422
Epoch: 3
loss:0.49630507826805115, [64 /938]
Accuracy of the current batch: 0.84375
loss:0.44981786608695984, [32064 /938]
Accuracy of the current batch: 0.84375
Final training accuracy: 0.84375
Final testing accuracy: 0.8881999850273132
Epoch: 4
loss:0.42429712414741516, [64 /938]
Accuracy 

In a more general way, if there are more layers, the model will need more training epochs in order to tune all the parameters. <br>
Therefore, it can be more convenient to start the training using a simpler model and then adding more and more hidden layers.

Alternatively, it is possible to implement a non-sequential structure that requires a specific layering of the neural network model, although, generally speaking, it may be more cumbersome compared to the sequential implementation. <br>
N.B.: Pytorch also allows to merge the two implementations.

In [ ]:
import torch # Pytorch works using tensors, which can be implemented as n-dimensional arrays similar to numpy arrays
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn # for creating the neural netowork layers
import torchmetrics

# Start by importing the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the training data of MNIST and into a 'data' folder
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor()) # (down)load, and convert to tensor, the testing data of MNIST and into a 'data' folder

# Define the structure of the MLP model through classes extending the Module class of nn.
device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility
class MyMLP(nn.Module):
    def __init__(self): # define the layers needed by the model
        super().__init__() # initialize the attributes of the parent class
        self.input_layer = nn.Linear(28*28, 15)
        self.activation_fn = nn.reLU()
        self.layer2 = nn-Linear(15, 15)
        self.output_layer = nn.Linear(15, 10)
        self.flatten = nn.Flatten() # flatten the structure into a one-dimensional array
    def forward(self, x): # specify how the data goes inside the model by connecting the layers
        x = self.flatten(x) # convert the input image into a one-dimensional array
        # Now, the goal is to pass the data into all the layers of the structure.
        x = self.input_layer(x)
        x = self.activation_fn(x)
        x = self.layer2(x)
        x = self.activation_fn(x)
        logits = self.output_layer(x) # it is also possible write all this in a single line of code, although it tends to be very cumbersome
        return logits

# Create an instance of the MLP model.
model = MyMLP()

# Start by defining the hyperparameters of the model.
epochs = 5 # check the whole dataset five times
batch_size = 64 # check 64 data samples at once
learning_rate = 0.0001 # use a smaller learning rate in order to obtain more accurate results during the backpropagation phase

# Define the loss function for the classification model.
loss_function = nn.CrossEntropyLoss() # choose cross entropy loss

# Use a dataloader for processing the data in batches.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # dataloader for the training dataset
test_dataloader = DataLoader(test_data, batch_size=batch_size) # dataloader for the testing dataset

# Create the accuracy metric for estimating the goodness of the model.
metric = torchmetrics.Accuracy(task='multiclass', num_classes=10)

# Define the optimizer for gradient descent.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) # use the AdamW model on the parameters

# Define the training loop for the training stage.
def training_loop(dataloader, model, loss_function, optimizer):
    dataset_size = len(dataloader)
    # Start by loading the data batch from the disk.
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X) # get the predictions of the model
        # Compute the prediction error using the loss function.
        loss = loss_function(pred, y)
        # Perform backpropagation in order to minimize future losses.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss:{loss}, [{current} /{dataset_size}]")
            acc = metric(pred, y)
            print(f"Accuracy of the current batch: {acc}")
    # Print the final training accuracy.
    acc = metric.compute()
    print(f"Final training accuracy: {acc}")
    metric.reset() # reset for future loops

# Define the testing loop for the testing stage.
def testing_loop(dataloader, loss_function, model):
    # Remember to disable the weights update.
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            acc = metric(pred, y)
    acc = metric.compute()
    print(f"Final testing accuracy: {acc}")
    metric.reset()

# Perform the actual model training and testing.
for e in range(epochs):
    print(f"Epoch: {e}")
    training_loop(train_dataloader, model, loss_function, optimizer)
    testing_loop(test_dataloader, loss_function, model)
print("Done")